# Sensibilidad del metodo bi-nivel

Metodo fijo: **bi-nivel por vendor**, zonas espacialmente compactas (orden de recorrido / snake en el Problema 1) y QAP por zona (Problema 2).

Se barre **una dimension por vez** alrededor de una configuracion base (merchant, jaccard, top_k(10), zona = swaps lam=0.5, snake), para ver la sensibilidad de las rutas a cada eleccion de bloque.

In [ ]:
# Setup: datos, geometria fija y co-ocurrencia (calculada una sola vez)
import numpy as np, pandas as pd
from abs_affinity_based_slotting.config import RAW_DIR
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.demand import build_cooccurrence, build_sku_demand, affinity_registry, filter_registry
from abs_affinity_based_slotting.warehouse import occupied_locations, build_location_costs, build_bay_distance_matrix
from abs_affinity_based_slotting.slotting import build_instance
from abs_affinity_based_slotting.clustering import clustering_registry
from abs_affinity_based_slotting.methods import (CurrentSlotting, DemandGreedySlotting,
    LinearAssignmentSlotting, SwapSearchSlotting, BiLevelSlotting)
from abs_affinity_based_slotting.evaluation import Evaluator

ds = WarehouseDataLoader(RAW_DIR).load_all()
split = split_picking_events(ds.picking_events, test_size=0.2)
universe   = occupied_locations(ds.initial_stock)['sku'].to_numpy()
sku_demand = build_sku_demand(split.train)                       # f
loc_costs  = build_location_costs(ds.initial_stock, ds.distances)   # c
bay_dist   = build_bay_distance_matrix(ds.distances)             # D
evaluator  = Evaluator.from_tables(ds.coordinates, ds.distances, ds.initial_stock)

# La co-ocurrencia no depende de la metrica ni del filtro -> se calcula una vez.
co = build_cooccurrence(split.train, skus=universe)

In [ ]:
# Helper 1: arma la instancia eligiendo metrica de afinidad y filtro.
def make_instance(metric='jaccard', filter_name='top_k', k=10):
    A = affinity_registry.get(metric)().build(co.matrix, co.support, co.n_batches)
    A = filter_registry.get(filter_name)(k=k).filter(A)
    return build_instance(sku_demand, loc_costs, bay_dist,
                          initial_stock=ds.initial_stock, skus=universe, affinity=A)

# Instancia base (jaccard + top_k(10)).
instance_base = make_instance()

# Posicion en el recorrido (snake): solo depende de la geometria, no de la
# afinidad, asi que sirve para cualquier instancia (mismas ubicaciones/bays).
coord = ds.coordinates.set_index('bay_id')
bay_key = coord['aisle'].astype(float) * 1000.0 + coord['bay_number'].astype(float)
bay_key = np.nan_to_num(bay_key.reindex(instance_base.bay_ids).to_numpy(),
                        nan=np.nanmax(bay_key.values) + 1.0)
snake_pos = bay_key[instance_base.location_bay]

# Helper 2: corre un metodo y lo evalua en test.
def run(name, method, instance):
    sol = method.solve(instance)
    m = evaluator.evaluate(sol, split.test)
    return {'variante': name, 'mean': round(m.mean_batch_distance), 'p95': round(m.p95_batch_distance)}

## Baselines de referencia

`linear_assignment` global se omite a proposito: es O(n^3) (~40 min sobre 27k). `demand_greedy` alcanza practicamente el optimo lineal (lam=1) y es la referencia practica (ver notebook 02).

In [ ]:
ref = [
    run('current', CurrentSlotting(ds.initial_stock), instance_base),
    run('demand_greedy (ref lam=1)', DemandGreedySlotting(), instance_base),
]
pd.DataFrame(ref)

## Barrido 1 - metrica de afinidad

Base: merchant, top_k(10), zona = swaps lam=0.5, snake. Se cambia solo como se mide la afinidad: `jaccard`, `cosine`, `cooccurrence` (conteo crudo).

In [ ]:
clu = clustering_registry.get('merchant')()
rows = []
for metric in ['jaccard', 'cosine', 'cooccurrence']:
    inst = make_instance(metric=metric, filter_name='top_k', k=10)
    rows.append(run(f'afinidad={metric}',
                    BiLevelSlotting(clu, SwapSearchSlotting(lam=0.5), location_cost=snake_pos), inst))
pd.DataFrame(rows)

## Barrido 2 - filtro de afinidad

Base: merchant, jaccard, zona = swaps lam=0.5, snake. `top_k` (union) conserva el top-k por producto; `mutual_top_k` (interseccion) solo aristas mutuas (mas estricto).

In [ ]:
rows = []
for fname in ['top_k', 'mutual_top_k']:
    inst = make_instance(metric='jaccard', filter_name=fname, k=10)
    rows.append(run(f'filtro={fname}',
                    BiLevelSlotting(clu, SwapSearchSlotting(lam=0.5), location_cost=snake_pos), inst))
pd.DataFrame(rows)

## Barrido 3 - agrupamiento

`merchant` (vendor) vs `demand_class` (tiers de demanda). Solver de zona = `demand_greedy` (rapido) para aislar el efecto del agrupamiento y evitar el cluster gigante de `demand_class` (~18k) en solvers pesados.

In [ ]:
rows = []
for cname in ['merchant', 'demand_class']:
    rows.append(run(f'clustering={cname}',
                    BiLevelSlotting(clustering_registry.get(cname)(), DemandGreedySlotting(),
                                    location_cost=snake_pos), instance_base))
pd.DataFrame(rows)

## Barrido 4 - solver de zona y peso lambda

Cuanto aporta optimizar con **afinidad dentro de la zona de vendor**. Base: merchant, jaccard, top_k(10), snake. `zona=linear` ignora la afinidad (lam=1); `zona=swaps` la usa con peso creciente (lam menor = mas afinidad).

In [ ]:
rows = [run('zona=linear (lam=1)',
            BiLevelSlotting(clu, LinearAssignmentSlotting(), location_cost=snake_pos), instance_base)]
for lam in [0.7, 0.5, 0.3]:
    rows.append(run(f'zona=swaps lam={lam}',
                    BiLevelSlotting(clu, SwapSearchSlotting(lam=lam), location_cost=snake_pos), instance_base))
pd.DataFrame(rows)

## Como leer los resultados

- `mean` / `p95`: distancia de ruta por batch en test (menor es mejor).
- Referencia: `demand_greedy`. El bi-nivel **gana** si baja por debajo de el.
- Barrido 4 responde la pregunta central del plan vendor: si los `swaps` no bajan respecto de `zona=linear`, la afinidad intra-vendor aporta poco (esperable, porque merchant agrupa por vendor, no por co-picking).